## 1. 뉴스제목 가져오기
* user-agent 요청헤더를 반드시 설정해야 한다.

In [1]:
# requests 라이브러리 설치여부 확인
!pip show requests

Name: requests
Version: 2.33.1
Summary: Python HTTP for Humans.
Home-page: 
Author: 
Author-email: Kenneth Reitz <me@kennethreitz.org>
License: Apache-2.0
Location: C:\Users\vega2\anaconda3\Lib\site-packages
Requires: certifi, charset_normalizer, idna, urllib3
Required-by: anaconda-catalogs, anaconda-client, anaconda-cloud-auth, anaconda-project, conda, conda-build, conda-repo-cli, conda_package_streaming, cookiecutter, datashader, google-api-core, huggingface-hub, jupyterlab_server, langchain-classic, langchain-community, langchain-upstage, langsmith, panel, requests-file, requests-toolbelt, Sphinx, streamlit, tiktoken, tldextract, webdriver-manager, yarg


In [2]:
# beautifulsoup4 라이브러리 설치여부 확인
!pip show beautifulsoup4

Name: beautifulsoup4
Version: 4.13.5
Summary: Screen-scraping library
Home-page: https://www.crummy.com/software/BeautifulSoup/bs4/
Author: 
Author-email: Leonard Richardson <leonardr@segfault.org>
License: MIT License
Location: C:\Users\user\anaconda3\Lib\site-packages
Requires: soupsieve, typing-extensions
Required-by: conda-build, nbconvert


In [3]:
# reqeusts, bs4 import
import requests
import bs4
# BeautifulSoup 클래스 import
from bs4 import BeautifulSoup

In [4]:
# requests, bs4 버전 확인하기
print(f'requests 버전 = {requests.__version__}')
print(f'bs4 버전 = {bs4.__version__}')

requests 버전 = 2.33.1
bs4 버전 = 4.12.3


### 1. 뉴스 제목 추출하기

In [5]:
# IT/과학 뉴스 
#url = 'https://news.naver.com/section/105'

# dict타입으로 요청 파라미터 설정
req_param = {
    'sid': 105
}
# 
url = 'https://news.naver.com/section/{sid}'.format(**req_param)
print(url)

# 요청 헤더 설정 : 브라우저 정보 ( 사람처럼 보이게 하기 위함 )
req_header = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

# requests 의 get() 함수 호출하기 
res = requests.get(url, headers=req_header)
print(res.status_code)
print(res.ok)
print(type(res))
#print(res.text)
# 응답(response)이 OK 이면
# 응답 (response)에서 text 추출
# BeautifulSoup 객체 생성  
if res.ok:
    soup = BeautifulSoup(res.text,'html.parser')
    print(len(soup.select("div.sa_text a[href*='https://n.news.naver.com/mnews/']")))
    # CSS 선택자를 사용해서 a tag 목록 가져오기
    a_tags = soup.select("div.sa_text a[href*='https://n.news.naver.com/mnews/']")
    print(type(a_tags), type(a_tags[0])) # [Tag,Tag]
    # <a> 태그 리스트 순회하기    
    for a_tag in a_tags:
        title = a_tag.text.strip()
        link = a_tag['href']
        print(title, link)
else:
    # 응답(response)이 Error 이면 status code 출력    
    print(f'Error Code = {res.status_code}')



https://news.naver.com/section/105
200
True
<class 'requests.models.Response'>
88
<class 'bs4.element.ResultSet'> <class 'bs4.element.Tag'>
네이버, 남미 노린다…브라질·칠레 AI 기업과 협업 논의 https://n.news.naver.com/mnews/article/215/0001261121
 https://n.news.naver.com/mnews/article/comment/215/0001261121
우주청, 첫 태양권 탐사 구상 공개…NASA·ESA와 협력 '시동' https://n.news.naver.com/mnews/article/138/0002236528
 https://n.news.naver.com/mnews/article/comment/138/0002236528
"AI 혁신 속도낸다" ... 클래시스, 삼성전자 출신 김택수 CTO 선임 https://n.news.naver.com/mnews/article/015/0005317226
 https://n.news.naver.com/mnews/article/comment/015/0005317226
삼성 갤럭시 Z8 시리즈, 사전 판매 144만 대 ‘역대 최다’ https://n.news.naver.com/mnews/article/422/0000892138
 https://n.news.naver.com/mnews/article/comment/422/0000892138
LG유플러스, 보안전문기업 파고네트웍스 인수… 24시간 보안 대응 https://n.news.naver.com/mnews/article/025/0003542176
 https://n.news.naver.com/mnews/article/comment/025/0003542176
원안위, 고리 3·4호기 계속운전 하반기 심의…월성 은폐 처벌도 https://n.news.naver.com/mnews/article/421/0009096825


### 1.1 뉴스제목 추출하는 함수 선언하기

In [35]:
import requests
from bs4 import BeautifulSoup

#section_dict = {100:'정치',101:'경제',102:'사회',103:'생활/문화',104:'세계',105:'IT/과학'}
section_dict = {'정치':100,'경제':101,'사회':102,'생활/문화':103,'세계':104,'IT/과학':105}

def print_news(section_name):  #print_new('생활/문화') 
    sid = section_dict.get(section_name,'정치')
    url = f'https://news.naver.com/section/{sid}'
    print(f'{section_name} 뉴스 {url}')
    req_header = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
    }
    res = requests.get(url, headers=req_header)
    if res.ok:
        soup = BeautifulSoup(res.text,'html.parser')
        # CSS 선택자를 사용해서 a tag 목록 가져오기
        a_tags = soup.select("div.sa_text a[href*='https://n.news.naver.com/mnews/']")
        # <a> 태그 리스트 순회하기    
        for a_tag in a_tags:
            title = a_tag.text.strip()
            link = a_tag['href']
            print(title, link)
    else:
        # 응답(response)이 Error 이면 status code 출력    
        print(f'Error Code = {res.status_code}')

In [ ]:
print_news('경제')

### 2. Image 다운로드
* referer 요청 헤더를 반드시 설정해야 한다.

In [ ]:
import requests
import os

# 육아일기 73회차
req_header = {
    'referer':'https://comic.naver.com/webtoon/detail?titleId=812354&no=208&week=sun',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

img_urls = [
    'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg',
    'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg',
    'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_3.jpg'
]

for img_url in img_urls:
    # requests 의 get(url, headers) 함수 호출하기 
    res = requests.get(img_url, headers=req_header)
    print(res.status_code)        
    # binary 응답 데이터 가져오기
    img_data = res.content    
    # url에서 파일명만 추출하기
    file_name = os.path.basename(img_url)
    print(file_name)        
    # binday data를 file에 write하기
    with open(file_name,'wb') as file:
        print(f'Writing to {file_name}({len(img_data):,} bytes)')
        file.write(img_data)


* 현재 요청된 페이지의 image 모두 다운로드 해보기

In [ ]:
import requests
from bs4 import BeautifulSoup
import os

webtoon_url = 'https://comic.naver.com/webtoon/detail?titleId=812354&no=208&week=thu'

req_header = {
    'referer':webtoon_url,
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

res = requests.get(webtoon_url, headers=req_header)
if res.ok:
    # .jpg 파일명을 추출해서 list에 저장하기
    soup = BeautifulSoup(res.text,'html.parser')
    print(len(soup.select("img[src*='IMAG01']")))
    img_tags = soup.select("img[src*='IMAG01']")
    # img_url_list = [] #list()
    # for img_tag in img_tags:
    #     img_url = img_tag['src']
    #     img_url_list.append(img_url)

    # List Comprehension        
    img_url_list2 = [img_tag['src'] for img_tag in img_tags]    
    print(img_url_list2[:2])

    imgdir_name = 'img'
    if not os.path.isdir(imgdir_name):
        os.mkdir(imgdir_name)

    for img_url in img_url_list2:
        # requests 의 get(url, headers) 함수 호출하기 
        res = requests.get(img_url, headers=req_header)
        # binary 응답 데이터 가져오기
        img_data = res.content    

        #img/xxxIMG01.jpg
        file_path = os.path.join(imgdir_name,os.path.basename(img_url))
        # binday data를 file에 write하기
        with open(file_path,'wb') as file:
            print(f'Writing to {file_path}({len(img_data):,} bytes)')
            file.write(img_data)        
else:
    print(f'Error Code = {res.status_code}')



#### 리팩토링 된 코드

In [ ]:
import requests
from bs4 import BeautifulSoup
import os

# 기본 설정
url = 'https://comic.naver.com/webtoon/detail?titleId=833255&no=3&week=tue'
req_header = {'referer': url}
imgdir_name = 'img'

# 이미지 저장 폴더가 없으면 생성 ( sub 디렉토리도 생성)
os.makedirs(imgdir_name, exist_ok=True)

# 웹 페이지 요청 및 확인
res = requests.get(url)
if not res.ok:
    print(f'Error Code = {res.status_code}')
    exit()

# 이미지 URL 추출
soup = BeautifulSoup(res.text, 'html.parser')
img_url_list = [img_tag['src'] for img_tag in soup.select("img[src*='IMAG01']")]

# 이미지 다운로드
for img_url in img_url_list:
    res = requests.get(img_url, headers=req_header)
    if res.ok:
        img_data = res.content
        file_path = os.path.join(imgdir_name, os.path.basename(img_url))
        with open(file_path, 'wb') as file:
            print(f'Writing to {file_path} ({len(img_data):,} bytes)')
            file.write(img_data)
    else:
        print(f'Error Code = {res.status_code} for {img_url}')

### 3. 파일 업로드 하기
* https://httpbin.org/
* http://httpbin.org/post 업로드 요청을 할 수 있는 url

In [ ]:
import requests

upload_files = {
    'img1': open('img/f1.jpg','rb'),
    'img2': open('img/f2.jpg','rb'),
}
#print(upload_files)

url = 'http://httpbin.org/post'
# file 업로드 하려면 requests의 post 함수에 files 속성을 사용합니다.
res = requests.post(url, files=upload_files)
print(res.status_code)
print(res.json()['files']['img1'])

### 4. 캡챠(이미지) API 호출하기
* urllib 사용
* 1. 캡차 키 발급 요청
  2. 캡차 이미지 요청
  3. 사용자 입력값 검증 요청

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
print(CLIENT_ID[:4])

CLIENT_SECRET = os.getenv("CLIENT_SECRET")
print(CLIENT_SECRET[:4])


SEQJ
qpOE


In [10]:
# 캡차 키 발급 요청
import urllib.request

code = "0"
url = "https://openapi.naver.com/v1/captcha/nkey?code=" + code

request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",CLIENT_ID)
request.add_header("X-Naver-Client-Secret",CLIENT_SECRET)

response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)

{"key":"KzZQcXSMtS0vjcal"}


In [11]:
# 캡차 이미지 요청
import urllib.request

key = "KzZQcXSMtS0vjcal" # 캡차 Key 값
url = "https://openapi.naver.com/v1/captcha/ncaptcha.bin?key=" + key
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",CLIENT_ID)
request.add_header("X-Naver-Client-Secret",CLIENT_SECRET)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    print("캡차 이미지 저장")
    response_body = response.read()
    with open('img/captcha.jpg', 'wb') as f:
        f.write(response_body)
else:
    print("Error Code:" + rescode)

캡차 이미지 저장


In [12]:
#  사용자 입력값 검증 요청
import requests

code = "1"
key = "KzZQcXSMtS0vjcal"
value = "3EF9U"

req_header = {
    "X-Naver-Client-Id": CLIENT_ID,
    "X-Naver-Client-Secret": CLIENT_SECRET
}
req_param = {
    "code": code,
    "key": key,
    "value": value
}

url = "https://openapi.naver.com/v1/captcha/nkey"

try:
    res = requests.get(url, headers=req_header, params=req_param)
    #4xx,5xx 오류가 발생하면 예외 발생시킴
    res.raise_for_status()

    print(res.text)
except requests.exceptions.RequestException as e:
    print(f'Status Code : {res.status_code}')
    print(f'Error 발생 : {e} ')

{"result":true,"responseTime":39.8}


* requests를 사용하는 코드로 변경하기
* [requests docs](https://requests.readthedocs.io/en/latest/user/quickstart/)

### 5. 블로그 검색하기

In [3]:
import requests
from pprint import pprint

headers = {
    'X-Naver-Client-Id': CLIENT_ID,
    'X-Naver-Client-Secret': CLIENT_SECRET,
}

payload = {
    'query': '파이썬',
    'display': 100,
    'sort': 'sim'
}

url = 'https://openapi.naver.com/v1/search/blog.json'

# requests get(url, params, headers) 요청 
res = requests.get(url, params=payload, headers=headers)

# json() 함수로 응답 결과 가져오기 [{},{},{}]
# 'title' , 'bloggername' , 'description' , 'bloggerlink' , 'link'
print(len(res.json()['items'])) 
pprint(res.json()['items'])

items_data = res.json()['items']
# 'title' , 'bloggername' , 'description' , 'bloggerlink' , 'link'



100
[{'bloggerlink': 'blog.naver.com/lmwu123',
  'bloggername': '쉼표, 느낌표',
  'description': '그렇게 이것저것 알아보다가 가장 많이 추천받은 언어가 바로 <b>파이썬</b>이었어요! 오늘은 '
                 '계양구컴퓨터학원에서 <b>파이썬</b> 기초 수업 들은 후기를 남겨보려고 합니다 :-) 첫 프로그래밍 언어로 '
                 '<b>파이썬</b>을 선택한 이유... ',
  'link': 'https://blog.naver.com/lmwu123/224325518909',
  'postdate': '20260624',
  'title': '계양구컴퓨터학원 왕초보 <b>파이썬</b> 기초 수업'},
 {'bloggerlink': 'blog.naver.com/blackhole_chinese',
  'bloggername': '묘한 교육 연구소',
  'description': '<b>파이썬</b> 자격증 시험 종류 및 공부방법 취득후기 <b>파이썬</b> 자격증 취득자의 후기를 '
                 '소개합니다. 처음부터 개발자가 되고 싶은 건 아니었어요! 취업을 준비하다 보니 자연스레 눈길이 갔죠! '
                 '문과로는 도무지... ',
  'link': 'https://blog.naver.com/blackhole_chinese/224107596612',
  'postdate': '20251212',
  'title': '<b>파이썬</b> 자격증 시험 종류 및 공부방법 취득후기'},
 {'bloggerlink': 'blog.naver.com/ureka310',
  'bloggername': '아빠 집으로 가는 길',
  'description': '단순 대전<b>파이썬</b>학원 대전자바학원처럼 단과 과정이 아닌, 취업을 목적으로 하는 IT 실무 기술을 '
                 '배우는... 대전<b>파이썬

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
print(CLIENT_ID[:4])

CLIENT_SECRET = os.getenv("CLIENT_SECRET")
print(CLIENT_SECRET[:4])

ic0k
An89


In [2]:
import requests
from pprint import pprint

headers = {
    'X-NCP-APIGW-API-KEY-ID': CLIENT_ID,
    'X-NCP-APIGW-API-KEY': CLIENT_SECRET,
}

payload = {
    'query': '파이썬',
    'display': 100,
    'sort': 'sim'
}
domain_url = 'https://naverapihub.apigw.ntruss.com'
url = f'{domain_url}/search/v1/blog'

# requests get(url, params, headers) 요청 
res = requests.get(url, params=payload, headers=headers)
# json() 함수로 응답 결과 가져오기 [{},{},{}]
print(len(res.json()['items'])) 
pprint(res.json()['items'])

items_data = res.json()['items']
# 'title' , 'bloggername' , 'description' , 'bloggerlink' , 'link'


100
[{'bloggerlink': 'blog.naver.com/lmwu123',
  'bloggername': '쉼표, 느낌표',
  'description': '그렇게 이것저것 알아보다가 가장 많이 추천받은 언어가 바로 <b>파이썬</b>이었어요! 오늘은 '
                 '계양구컴퓨터학원에서 <b>파이썬</b> 기초 수업 들은 후기를 남겨보려고 합니다 :-) 첫 프로그래밍 언어로 '
                 '<b>파이썬</b>을 선택한 이유... ',
  'link': 'https://blog.naver.com/lmwu123/224325518909',
  'postdate': '20260624',
  'title': '계양구컴퓨터학원 왕초보 <b>파이썬</b> 기초 수업'},
 {'bloggerlink': 'blog.naver.com/blackhole_chinese',
  'bloggername': '묘한 교육 연구소',
  'description': '<b>파이썬</b> 자격증 시험 종류 및 공부방법 취득후기 <b>파이썬</b> 자격증 취득자의 후기를 '
                 '소개합니다. 처음부터 개발자가 되고 싶은 건 아니었어요! 취업을 준비하다 보니 자연스레 눈길이 갔죠! '
                 '문과로는 도무지... ',
  'link': 'https://blog.naver.com/blackhole_chinese/224107596612',
  'postdate': '20251212',
  'title': '<b>파이썬</b> 자격증 시험 종류 및 공부방법 취득후기'},
 {'bloggerlink': 'blog.naver.com/ureka310',
  'bloggername': '아빠 집으로 가는 길',
  'description': '단순 대전<b>파이썬</b>학원 대전자바학원처럼 단과 과정이 아닌, 취업을 목적으로 하는 IT 실무 기술을 '
                 '배우는... 대전<b>파이썬

In [7]:
import requests
from pprint import pprint

headers = {
    'X-NCP-APIGW-API-KEY-ID': CLIENT_ID,
    'X-NCP-APIGW-API-KEY': CLIENT_SECRET,
}

payload = {
    'query': '파이썬',
    'display': 100,
    'sort': 'sim'
}
domain_url = 'https://naverapihub.apigw.ntruss.com'
url = f'{domain_url}/search/v1/blog'
print(url)

# requests get(url, params, headers) 요청 
res = requests.get(url, params=payload, headers=headers)
# json() 함수로 응답 결과 가져오기 [{},{},{}]
print(len(res.json()))
print(res.json()) 
#pprint(res.json()['items'])

#items_data = res.json()['items']

https://naverapihub.apigw.ntruss.com/search/v1/blog
5
{'lastBuildDate': 'Thu, 06 Aug 2026 19:28:18 +0900', 'total': 602653, 'start': 1, 'display': 100, 'items': [{'title': '계양구컴퓨터학원 왕초보 <b>파이썬</b> 기초 수업', 'link': 'https://blog.naver.com/lmwu123/224325518909', 'description': '그렇게 이것저것 알아보다가 가장 많이 추천받은 언어가 바로 <b>파이썬</b>이었어요! 오늘은 계양구컴퓨터학원에서 <b>파이썬</b> 기초 수업 들은 후기를 남겨보려고 합니다 :-) 첫 프로그래밍 언어로 <b>파이썬</b>을 선택한 이유... ', 'bloggername': '쉼표, 느낌표', 'bloggerlink': 'blog.naver.com/lmwu123', 'postdate': '20260624'}, {'title': '<b>파이썬</b> 자격증 시험 종류 및 공부방법 취득후기', 'link': 'https://blog.naver.com/blackhole_chinese/224107596612', 'description': '<b>파이썬</b> 자격증 시험 종류 및 공부방법 취득후기 <b>파이썬</b> 자격증 취득자의 후기를 소개합니다. 처음부터 개발자가 되고 싶은 건 아니었어요! 취업을 준비하다 보니 자연스레 눈길이 갔죠! 문과로는 도무지... ', 'bloggername': '묘한 교육 연구소', 'bloggerlink': 'blog.naver.com/blackhole_chinese', 'postdate': '20251212'}, {'title': '대전<b>파이썬</b>학원 자바 실무 중심 대전IT학원', 'link': 'https://blog.naver.com/ureka310/224120952595', 'description': '단순 대전<b>파이썬</b>학원

In [7]:
# data/blog.json 파일 생성하기
import json

with open('data/blog.json','w', encoding='utf-8') as file:
    json.dump(items_data, file)

In [8]:
import pandas as pd

print(pd.__version__)

2.2.2


In [9]:
data = pd.read_json('data/blog.json')
print(data.shape)

(100, 6)


In [10]:
data.head()

,title,link,description,bloggername,bloggerlink,postdate
0,계양구컴퓨터학원 왕초보 <b>파이썬</b> 기초 수업,https://blog.naver.com/lmwu123/224325518909,그렇게 이것저것 알아보다가 가장 많이 추천받은 언어가 바로 <b>파이썬</b>이었어...,"쉼표, 느낌표",blog.naver.com/lmwu123,20260624
1,<b>파이썬</b> 자격증 시험 종류 및 공부방법 취득후기,https://blog.naver.com/blackhole_chinese/22410...,<b>파이썬</b> 자격증 시험 종류 및 공부방법 취득후기 <b>파이썬</b> 자격...,묘한 교육 연구소,blog.naver.com/blackhole_chinese,20251212
2,대전<b>파이썬</b>학원 자바 실무 중심 대전IT학원,https://blog.naver.com/ureka310/224120952595,"단순 대전<b>파이썬</b>학원 대전자바학원처럼 단과 과정이 아닌, 취업을 목적으로...",아빠 집으로 가는 길,blog.naver.com/ureka310,20251224
3,재직자 내일배움카드 후기｜비전공자 <b>파이썬</b>·생성형 AI...,https://blog.naver.com/silverjudy-/224295662808,위한 <b>파이썬</b>과 생성형 AI 디지털업무 자동화 기초과정” 이었다. 수강한...,"우리는 너무 많이 생각하고, 너무 적게 느낀다.",blog.naver.com/silverjudy-,20260525
4,<b>파이썬</b>자격증 취업 장소 및 공부방법,https://blog.naver.com/togyu911/223954106266,<b>파이썬</b>자격증 취업 장소 및 공부방법 제가 공부한 솔직한 후기를 바탕으로...,IIS 지식정보네트워크,blog.naver.com/togyu911,20250731


In [23]:
# bloggername 칼럼의 값을 가져오기
data['bloggername'].unique()

array(['묘한 교육 연구소', '아빠 집으로 가는 길', '더불어 사는 세상 이도형', '쉼표, 느낌표', '소소한 일상?',
       'IIS 지식정보네트워크', '지혜를 여는 부엉이상점', 'Leather', '진상의 추억', '모니토리 IT 이야기',
       'KG에듀원 아이티뱅크 공식블로그', 'No.1 전자엔지니어 전문몰 아이씨뱅큐', '카리스마',
       'A.I Researcher - Ph.D 인공지능교육', '소라윙즈의 소소한 IT/게임 리뷰', '엔돌슨의 IT이야기',
       '주들의 그리 대단하진 않지만 굉장한 이야기', 'MJ의 방방곡곡',
       '시원한 파랑_"시작하시죠, 파란만장한 컴퓨터 인생"', '에릭의 IT스토리', '서울에 사는 한 여자',
       '테크헌터 IT 팩토리', 'WORKS ON WEB & JP CULTURE', '버들붕어 스토리',
       '한국기술교육대학교 공식 블로그', '포토그래퍼 신남의 IT세상', '코드트리 공식 블로그', 'ෆ 일상의 기록 ෆ',
       'Chaconne', '미래세상', 'IT국비지원 – 왕초보에서 취업까지', 'ENTJ의 솔직한 일상',
       '고려아카데미컨설팅', '.', '꽃은 죽어도 씨앗은 남는다고', '던창 해군 Herimon의 개인세계',
       '여행백과사진', '조용한 기록실', "keukrae's blog", '해커스 유학 블로그',
       'Clarke Square', '여행사 출신의 육아여행기', '1800-1773 컴스마일', '반짝이는 하루',
       '세수하면이병헌 IT/자동차', '왕구렁이 블로그', '파푸리카', '경상국립대학교 공식 블로그',
       '아이들과미래재단 공식블로그', '성북문화재단', '티파의 게임 라이프',
       '일과 학업을 동시에! 한국열린사이버대학교!', '김동균의 소프트웨어아카데미(2004~ )',
       '처리씨 ⓙ ⓗ ⓒ 3 0 4 4', '한